## 1. torch_prep_kfold.py --initial_split  → writes *_trn_final.csv, *_tst_preprocess.csv

In [114]:
import sys
import logging
import argparse
import numpy as np
import pandas as pd
from typing import Any, Optional, List, Tuple
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import os

In [115]:
random_state = 42
test_percentage = 0.20
np.random.seed(42)

In [116]:
# torch_prep_kfold.py
# 
###########################################################################
# 1) INITIAL SPLIT
###########################################################################

In [117]:
id_col = 'sequence'
label_col = 'bind_avg'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

                                 sequence  bind_avg
0    GAGGAAGCAGCCCTCGCCCCTGTCGGTGGAAAGAAG -0.758634
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG -1.003319
2    ATCTGATCAAAACAACGAATTCCAAAACAAAGTAAT -0.800322
3    CCAATATTCCTTTGTGAGACCCTCCACAAATGCTAA -0.941242
4    GAGGACGCGAACCGGCACGCTGCGCCTTTAAGGAGT -0.684116
..                                    ...       ...
163  ACATAGGGACGGGGCCATGCGGTGGGCGGGTGGAAC  1.074979
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT  1.152711
165  CGCGGAGACCCGAAGCACGTGGTATCCATACTAGTT  2.121860
166  GCCCCCGACCCCGCGCACGCGGCCCCGCCCCGCGCG  1.105868
167  ATTAGCCAAACTAAACACGTGTATTGATTTTAGATG  1.884407

[168 rows x 2 columns]


In [118]:
usecols = ['sequence','run','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data)

                                    sequence  run  VDWAALS       EEL  \
0       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830   
1       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424   
2       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687   
3       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573   
4       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268   
...                                      ...  ...      ...       ...   
272155  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -148.291 -1941.978   
272156  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -142.375 -1959.709   
272157  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -167.433 -1911.650   
272158  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -136.663 -1920.439   
272159  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -146.999 -1929.330   

             EGB   ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0       1841.253 -36.482  -1.940432         -165.447020 -1.655

### Sanity Check

Checking if we retain random columns before and after merge

In [119]:
search_feat_row = feature_data[(feature_data['sequence'] == 'CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT') & (feature_data['VDWAALS'] == -207.264)]
print(search_feat_row)

                                    sequence  run  VDWAALS      EEL      EGB  \
105492  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.57  1908.59   

         ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  
105492 -34.442  -3.478696         -140.468722     -5.839134     -22.871906  


In [120]:
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830  1841.253   
1  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424  1835.847   
2  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687  1851.589   
3  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573  1814.002   
4  GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268  1847.934   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -36.482  -1.940432         -165.447020 -1.655191e-03     -26.046553   
1 -36.023  -2.003962         -155.422935 -4.708262e-02     -24.150637   
2 -35.802  -2.269901         -142.386371 -5.901517e-29     -24.329875   
3 -34.799  -2.838678         -147.918585 -3.236084e-07     -23.615145   
4 -34.391  -2.810414         -151.012478 -1.784784e-05     -23.698348   

   bind_avg  
0  1.531336  
1  1.531336  
2  1.531336  
3  1.531336  
4  1.531336   (272160, 11)


In [121]:
search_one_row = df_merged[(df_merged['sequence'] == 'CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT') & (df_merged['VDWAALS'] == -207.264)]
print(search_one_row)


                                    sequence  run  VDWAALS      EEL      EGB  \
105492  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.57  1908.59   

         ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
105492 -34.442  -3.478696         -140.468722     -5.839134     -22.871906   

        bind_avg  
105492   1.97632  


In [122]:
# Shuffle everything
df_merged = df_merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT    7 -207.264 -1958.570  1908.590   
1  TCCTAAACAGGAAGCCATGAGGTGAGCAGAGACACT    2 -229.177 -1917.175  1870.986   
2  TTAGAAAAATAGTTTAAAATCTAGAGTTAATTAACC    3 -197.506 -1830.441  1787.958   
3  CCAGCTCTCCACCGCCGCGTGCGCCTGCAGACGCTC    1 -207.553 -1891.794  1845.707   
4  CCCCCAGCGCTCCGGCACGCGCCGGGAGACCTCCGG   19 -201.484 -1923.635  1874.505   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -34.442  -3.478696         -140.468722     -5.839134     -22.871906   
1 -32.215 -18.599640         -143.435504     -1.459304     -22.335776   
2 -29.223  -2.514019         -120.507243     -4.043271     -18.798981   
3 -31.909 -13.343306         -138.470452     -3.535617     -22.011054   
4 -31.059  -4.628190         -138.103395     -0.006414     -22.886912   

   bind_avg  
0  1.976320  
1 -0.061198  
2  0.002694  
3  0.153350  
4 -0.483023   (272160, 11)


In [123]:
# Split train vs test by sequence or stratified group
## for regression, not bin or mclass

from sklearn.model_selection import GroupKFold

unique_seqs = df_merged[id_col].unique() #unique sequences are extracted
np.random.seed(random_state)
np.random.shuffle(unique_seqs) #randomly shuffles unique sequences

n_train = int((1 - test_percentage) * len(unique_seqs)) # Compute train/test split boundary

train_seqs = unique_seqs[:n_train] # First 85% (after shuffle) = training sequence IDs.
test_seqs  = unique_seqs[n_train:]

# Filter the rows accordingly
df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print(df_train.shape, df_test.shape)


(217080, 11) (55080, 11)


In [124]:
if "run" in df_train.columns:
    df_train.drop(columns=["run"], inplace=True, errors="ignore")
if "run" in df_test.columns:
    df_test.drop(columns=["run"], inplace=True, errors="ignore")

print(df_train.shape, df_test.shape)

(217080, 10) (55080, 10)


In [125]:
# Save
train_file = f"reg_trn_final.csv"
test_file  = f"reg_tst_preprocess.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

In [126]:
# torch_prep_kfold.py
# 
###########################################################################
# 2) PROCESS MODE: "train" (or "test" later)
###########################################################################

In [127]:
if "run" in df_train.columns:
    print("Column exists!")
else:
    print("run column not present")

run column not present


In [128]:
# Keep only the last X% if requested
keep_last_percent = 90

"""
Retain only the last keep_percent fraction of rows in each sequence group.
If 'run' column exists, sort by it first.
"""

## training dataframe is df_train, or saved as reg_trn_final.csv

df_sorted = df_train.copy()

group_sizes = df_sorted.groupby(id_col)[id_col].transform("size")
cumcount = df_sorted.groupby(id_col).cumcount()

n_keep = (group_sizes * (keep_last_percent / 100.0)).astype(int)
n_keep = n_keep.mask(n_keep < 1, 1)  # ensure at least 1 row if fraction>0
mask = cumcount >= (group_sizes - n_keep)
df_train = df_sorted[mask].reset_index(drop=True)


In [132]:
# Average numeric features in chunks of size --navg

navg = 280

def average_features_for_sequence(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Shuffle the sequence's rows, chunk into size navg, and average numeric features.
    Label = label from the first row of each chunk.
    """
    results = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in [id_col, label_col]]
    df_shuffled = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    n_chunks = len(df_shuffled) // navg
    if n_chunks < 1:
        return pd.DataFrame(columns=df.columns)
    for i in range(n_chunks):
        chunk = df_shuffled.iloc[i * navg : (i + 1) * navg]
        row_dict = {col: chunk[col].mean() for col in feature_cols}
        row_dict[label_col] = chunk[label_col].iloc[0]
        row_dict[id_col] = chunk[id_col].iloc[0]
        results.append(row_dict)
    df_out = pd.DataFrame(results)
    col_order = [id_col] + sorted([c for c in df_out.columns if c != id_col])
    return df_out[col_order]

def average_features_for_mutants(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Apply the above averaging function per sequence group, then concat.
    """
    all_chunks = []
    for seq, group in df.groupby(id_col):
        chunk_df = average_features_for_sequence(group, navg, id_col, label_col, random_state)
        all_chunks.append(chunk_df)
    if not all_chunks:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(all_chunks, ignore_index=True)


In [133]:

df_train_avg = average_features_for_mutants(
    df_train,
    navg=navg,
    id_col=id_col,
    label_col=label_col,
    random_state=random_state
)

In [134]:
print(df_train_avg)

                                 sequence  Delta_Entropy          EEL  \
0    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT     -22.388645 -1884.799107   
1    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT     -22.472499 -1885.653418   
2    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT     -22.405204 -1887.012386   
3    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT     -22.561287 -1886.242043   
4    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT     -22.353870 -1883.346154   
..                                    ...            ...          ...   
665  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT     -20.399098 -1889.613082   
666  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT     -20.447855 -1884.616600   
667  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT     -20.198519 -1880.494950   
668  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT     -20.725894 -1885.671693   
669  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT     -20.447606 -1882.455536   

             EGB      ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0    1838.509686 -31.712225 -13.537878         

In [ ]:
def compute_mean_std(
    df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute mean/std for numeric columns (excluding ID and label).
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in numeric_cols:
        numeric_cols.remove(id_col)
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    means = [(c, df[c].mean()) for c in numeric_cols]
    stds  = [(c, df[c].std())  for c in numeric_cols]
    return (
        pd.DataFrame(means, columns=["colname", "mean"]),
        pd.DataFrame(stds, columns=["colname", "std"])
    )

def save_mean_std(mean_df: pd.DataFrame, std_df: pd.DataFrame, filename: str) -> None:
    merged = pd.merge(mean_df, std_df, on="colname")
    merged.to_csv(filename, index=False)

In [ ]:
# Compute and save mean/std

mean_df, std_df = compute_mean_std(df_train_avg, 'reg',id_col, label_col)

print(mean_df, std_df)

              colname         mean
0       Delta_Entropy   -22.020105
1                 EEL -1897.569489
2                 EGB  1849.399520
3               ESURF   -31.864962
4           HB Energy    -8.122823
5  Hydrophobic Energy  -138.656960
6        Pi-Pi Energy    -3.045031
7             VDWAALS  -205.575531               colname        std
0       Delta_Entropy   0.790056
1                 EEL   8.893254
2                 EGB   8.082749
3               ESURF   1.089981
4           HB Energy   5.479892
5  Hydrophobic Energy   7.637841
6        Pi-Pi Energy   0.753757
7             VDWAALS  10.054925


In [138]:
stats_file = f"reg_train_stats.csv"
save_mean_std(mean_df, std_df, stats_file)

In [139]:
def apply_standardization(
    df: pd.DataFrame,
    stats_df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> pd.DataFrame:
    df_std = df.copy()
    means = dict(zip(stats_df["colname"], stats_df["mean"]))
    stds  = dict(zip(stats_df["colname"], stats_df["std"]))
    for col in df_std.columns:
        if col in [id_col, label_col]:
            continue
        if col in means and col in stds:
            mu  = means[col]
            sigma = stds[col]
            if sigma == 0 or np.isnan(sigma):
                df_std[col] = df_std[col] - mu
            else:
                df_std[col] = (df_std[col] - mu) / sigma
    return df_std

def load_mean_std(filename: str) -> pd.DataFrame:
    if not os.path.isfile(filename):
        logging.error(f"Mean/Std file not found: {filename}")
        sys.exit(1)
    df_stats = pd.read_csv(filename)
    needed_cols = {"colname", "mean", "std"}
    if not needed_cols.issubset(df_stats.columns):
        logging.error(f"Mean/Std file missing columns. Found: {df_stats.columns.tolist()}")
        sys.exit(1)
    return df_stats

In [ ]:
# Apply standardization
df_train_std = apply_standardization(
    df_train_avg,
    load_mean_std(stats_file),
    'reg',
    id_col,
    label_col
)

## We applied the simple Z-score normalization

In [142]:
print(df_train_std)

                                 sequence  Delta_Entropy       EEL       EGB  \
0    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT      -0.466473  1.435963 -1.347293   
1    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT      -0.572609  1.339900 -1.287546   
2    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT      -0.487433  1.187091 -1.135452   
3    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT      -0.684992  1.273712 -1.235243   
4    AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT      -0.422457  1.599340 -1.550827   
..                                    ...            ...       ...       ...   
665  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT       2.051762  0.894657 -0.913529   
666  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT       1.990049  1.456485 -1.500364   
667  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT       2.305641  1.919943 -2.001012   
668  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT       1.638125  1.337845 -1.370985   
669  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT       1.990364  1.699485 -1.758403   

        ESURF  HB Energy  Hydrophobic E